# 函数式编程`Functional Programming`
- 函数式编程的一个特点就是，允许把函数本身作为参数传入另一个函数，还允许返回一个函数！
## 一、高阶函数`Higher-order function`
1. 函数本身可以指向变量 --- 即可将函数赋值给变量 `f = abs`，进而可以通过变量调用函数

In [1]:
f = abs
f(-10)

10

2. 函数名也是一个变量

In [2]:
# 示例，实际不能这样写
abs = 10
abs(-10)  # 赋值后就不能调用函数了

TypeError: 'int' object is not callable

#### 高阶函数定义
- 一个函数就可以接收另一个函数作为参数，这种函数就称之为高阶函数

In [1]:
# 最简单的一个高阶函数
def add(x, y, f):
    return f(x) + f(y)

print(add(-5, 6, abs))

11


### 1. map/reduce
2. **map**把f(x)作用在一个`序列`的每一个元素并把结果生成一个新的`序列`

In [2]:
# map 用法示例
def f(x):
    return x * x

r = map(f, [1, 2, 3, 4, 5, 6, 7, 8, 9])
list(r)   # Iterator是惰性序列，因此通过list()函数让它把整个序列都计算出来并返回一个list。

[1, 4, 9, 16, 25, 36, 49, 64, 81]

2. reduce
- 效果reduce( f, [ x1, x2, x3, x4 ] ) = f(f(f(x1, x2), x3), x4)
- 这里f 函数必须有两个参数

In [4]:
# reduce的简单示例 --- 把序列[1, 3, 5, 7, 9]变换成整数13579
from functools import reduce
def fn(x, y):
    return x * 10 + y

reduce(fn, [1, 3, 5, 7, 9])

13579

In [7]:
# 利用reduce和map可以写出把str ('13579')转换为int (13579)的函数：
from functools import reduce
def fn(x, y):
    return x * 10 + y

def char2num(s):
    digits = {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}
    return digits[s]

reduce(fn, map(char2num, '13579'))

13579

In [10]:
# 练习1
# 把用户输入的不规范的英文名字，变为首字母大写，其他小写的规范名字。输入：['adam', 'LISA', 'barT']，输出：['Adam', 'Lisa', 'Bart']：
def normalize(name):
    return name.lower().capitalize()   # 整体转小写，再把首字母大写
    
# 测试:
L1 = ['adam', 'LISA', 'barT']
L2 = list(map(normalize, L1))
print(L2)


['Adam', 'Lisa', 'Bart']


In [12]:
# 练习2
# 请编写一个prod()函数，可以接受一个list并利用reduce()求积：
from functools import reduce

def product(x, y):
    return x * y
    
def prod(L):
    return(reduce(product,L))

print('3 * 5 * 7 * 9 =', prod([3, 5, 7, 9]))
if prod([3, 5, 7, 9]) == 945:
    print('测试成功!')
else:
    print('测试失败!')


3 * 5 * 7 * 9 = 945
测试成功!


In [13]:
# 练习3
# 利用map和reduce编写一个str2float函数，把字符串'123.456'转换成浮点数123.456：
from functools import reduce

def char2num(s):
    # 单个字符转数字
    digits = {'0':0,'1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7,'8':8,'9':9}
    return digits[s]

def fn(x, y):
    # 拼接数字：1,2,3 → 123
    return x * 10 + y

def str2float(s):
    # 分割整数段、小数段
    if '.' in s:
        int_str, dec_str = s.split('.')
    else:
        int_str, dec_str = s, ''
    
    # 整数部分合并成数字
    integer = reduce(fn, map(char2num, int_str))
    # 小数部分合并成数字
    decimal = reduce(fn, map(char2num, dec_str)) if dec_str else 0
    # 小数除以 10^长度
    res = integer + decimal / (10 ** len(dec_str))
    return res


print('str2float(\'123.456\') =', str2float('123.456'))
if abs(str2float('123.456') - 123.456) < 0.00001:
    print('测试成功!')
else:
    print('测试失败!')


str2float('123.456') = 123.456
测试成功!


### 2. fliter()
- 用于过滤序列，接受一个**判断函数**和一个序列

In [1]:
# 示例1
# 在一个list中，删掉偶数，只保留奇数，可以这么写：
def is_odd(n):
    return n % 2 == 1

list(filter(is_odd, [1, 2, 3, 4, 5, 6, 9, 10, 15]))

[1, 3, 5, 9, 15]

In [2]:
# 示例2
# 把一个序列中的空字符串删掉，可以这么写：
def not_empty(s):
    return s and s.strip()

list(filter(not_empty, ['A', '', 'B', None, 'C', '  ']))
# 结果: ['A', 'B', 'C']

['A', 'B', 'C']

[用filter求素数](https://liaoxuefeng.com/books/python/functional/higher-order-function/filter/index.html)

In [3]:
# 1. 生成无限奇数生成器 _odd_iter()
def _odd_iter():
    n = 1
    while True:
        n = n + 2
        yield n
# 2. 筛选判断函数 _not_divisible(n)
def _not_divisible(n):
    return lambda x: x % n > 0
# 返回一个匿名函数：判断数字 x不能被 n 整除
# 3. 素数总生成器 primes()（核心）
def primes():
    yield 2                  # 第一个素数单独产出
    it = _odd_iter()         # 初始迭代器：3,5,7,9,11...奇数序列
    while True:
        n = next(it)         # 取出序列第一个数，一定是素数
        yield n              # 输出这个素数
        # 用filter过滤：剔除所有n的倍数，得到新迭代器覆盖it
        it = filter(_not_divisible(n), it)

# 调用
for n in primes():
    if n < 100:
        print(n)
    else:
        break

2
3
5
7
11
13
17
19
23
29
31
37
41
43
47
53
59
61
67
71
73
79
83
89
97


In [4]:
# 练习
# 回数是指从左向右读和从右向左读都是一样的数，例如12321，909。请利用filter()筛选出回数：
def is_palindrome(n):
    return str(n) == str(n)[::-1]

# 测试:
output = filter(is_palindrome, range(1, 1000))
print('1~1000:', list(output))
if list(filter(is_palindrome, range(1, 200))) == [1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 22, 33, 44, 55, 66, 77, 88, 99, 101, 111, 121, 131, 141, 151, 161, 171, 181, 191]:
    print('测试成功!')
else:
    print('测试失败!')


1~1000: [1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 22, 33, 44, 55, 66, 77, 88, 99, 101, 111, 121, 131, 141, 151, 161, 171, 181, 191, 202, 212, 222, 232, 242, 252, 262, 272, 282, 292, 303, 313, 323, 333, 343, 353, 363, 373, 383, 393, 404, 414, 424, 434, 444, 454, 464, 474, 484, 494, 505, 515, 525, 535, 545, 555, 565, 575, 585, 595, 606, 616, 626, 636, 646, 656, 666, 676, 686, 696, 707, 717, 727, 737, 747, 757, 767, 777, 787, 797, 808, 818, 828, 838, 848, 858, 868, 878, 888, 898, 909, 919, 929, 939, 949, 959, 969, 979, 989, 999]
测试成功!


### 3. sorted
1. 可以直接对list排序

In [5]:
sorted([36, 5, -12, 9, -21])

[-21, -12, 5, 9, 36]

2. sorted()也是高阶函数，可以接收一个key函数来实现自定义的排序

In [7]:
# 例如按绝对值大小排序：
sorted([36, 5, -12, 9, -21], key=abs) 
# key函数将作用于每个元素上，并根据key函数返回的结果进行排序

[5, 9, -12, -21, 36]

3. 字符串排序：默认按照ASCII的大小比较

In [8]:
# 给sorted传入key函数，即可实现忽略大小写的排序：
sorted(['bob', 'about', 'Zoo', 'Credit'], key=str.lower)

['about', 'bob', 'Credit', 'Zoo']

4. 反向排序可以传入第三个参数`reverse=True`：

In [10]:
sorted(['bob', 'about', 'Zoo', 'Credit'], key=str.lower, reverse=True)

['Zoo', 'Credit', 'bob', 'about']

In [11]:
# 练习
# 假设我们用一组tuple表示学生名字和成绩：

L = [('Bob', 75), ('Adam', 92), ('Bart', 66), ('Lisa', 88)]
# 请用sorted()对上述列表分别按名字排序：
def by_name(t):
    # TODO
    return t[0]
        

L2 = sorted(L, key=by_name)
print(L2)

[('Adam', 92), ('Bart', 66), ('Bob', 75), ('Lisa', 88)]


## 二、返回函数

- 函数内部还有一个函数，是一种**闭包**程序结构

In [14]:
# 如果不需要立刻求和，而是在后面的代码中，根据需要再计算怎么办？
def lazy_sum(*args):
    def sum():
        ax = 0
        for n in args:   # 内部函数sum可以引用外部函数lazy_sum的参数和局部变量           
            ax = ax + n
        return ax
    return sum
# 当我们调用lazy_sum()时，返回的并不是求和结果，而是求和函数：
f = lazy_sum(1, 3, 5, 7, 9)
f

<function __main__.lazy_sum.<locals>.sum()>

==请再注意一点，当我们调用lazy_sum()时，每次调用都会返回一个新的函数，即使传入相同的参数==

### 闭包
- **特点**：内层函数会一直保留对外层局部变量的引用，**不会**随着外层函数执行结束而销毁

In [16]:
# 拆解出错案例
def count():
    fs = []
    # 循环 i = 1、2、3
    for i in range(1, 4):
        # 定义函数f：返回 i*i
        def f():
            return i * i
        # 把函数装进列表
        fs.append(f)
    return fs

# 拿到三个函数
f1, f2, f3 = count()

**全过程分步走**
1. 循环阶段：只创建函数，绝不执行函数内部
一共循环 3 次：

    - i=1：创建函数 f，存入列表，f内部只是记下 “以后返回 i*i”，此刻不算结果
    - i=2：又创建新 f，存入列表
    - i=3：再次创建新 f，存入列表
    循环结束后，i最终定格为 3
2. 调用阶段：才执行函数内部代码

In [19]:
print(f1())
print(f2())
print(f3())

9
9
9


In [20]:
# 如何修正：把循环变量固定快照下来
# 思路：再套一层函数，立刻把当前 i 的值传进去，生成固定参数的闭包，锁住当下数值。
def count():
    fs = []
    for i in range(1, 4):
        # 外层接收参数n，立刻固化当前i的值
        def g(n):
            def f():
                return n * n
            return f
        # 调用g(i)，立刻传入当前i，生成独立的f
        fs.append(g(i))
    return fs

f1, f2, f3 = count()
print(f1()) # 1
print(f2()) # 4
print(f3()) # 9

1
4
9


返回闭包时牢记一点：==返回函数不要引用任何循环变量，或者后续会发生变化的变量==。

### nonlocal

In [21]:
def inc():
    x = 0
    def fn():
        # nonlocal x
        x = x + 1
        return x
    return fn

f = inc()
print(f()) # 1
print(f()) # 2

UnboundLocalError: cannot access local variable 'x' where it is not associated with a value

- 内层函数的x读取不到外层的x的声明，所以需要在fn()内部加一个`nonlocal x`的声明

In [23]:
# 练习
# 利用闭包返回一个计数器函数，每次调用它返回递增整数：
def createCounter():
    num = 0
    def counter():
        nonlocal num
        num = num + 1
        return num 
    return counter

# 测试:
counterA = createCounter()
print(counterA(), counterA(), counterA(), counterA(), counterA()) # 1 2 3 4 5
counterB = createCounter()
if [counterB(), counterB(), counterB(), counterB()] == [1, 2, 3, 4]:
    print('测试通过!')
else:
    print('测试失败!')


1 2 3 4 5
测试通过!


## 三、匿名函数`lambda`
格式：`lambda 参数列表 ：返回值`

In [ ]:
# 区别
lambda x:25   # 必须传入参数，但永远返回25
lambda :25    # 不用传参，调用必须写()

In [ ]:
# 示例
lambda x:x * x
# 相当于
def f(x):
    return x * x

1. 匿名函数也是一个函数对象，也可以把匿名函数赋值给一个变量，再利用变量来调用该函数
2. 同样，也可以把匿名函数作为返回值返回

In [5]:
# 赋值给变量
f = lambda x: x * x
f

# 作为返回值返回
def build(x, y):
    return lambda: x * x + y * y

## 四、装饰器
- 补充知识：函数对象有一个__name__属性（注意：是前后各两个下划线），可以拿到函数的名字


In [7]:
def now():
    print('2024-6-1')
     
now.__name__

'now'

### 一、本质
- 装饰器 = **用来给函数增加额外功能、但不修改原函数代码、不改动原函数调用方式的工具**
- 底层：**装饰器本质是接收函数作为参数、返回新函数的高阶函数**

### 二、示例
- 要求：写一个计时装饰器，统计函数运行耗时

In [8]:
import time

# 装饰器 --- 参数是原函数，调用后原函数名指向装饰后的新函数
def timer(func):
    # 包装函数
    def wrapper():
        start = time.time()  # 时间戳写法
        func()  # 执行原函数
        end = time.time()
        print(f"运行耗时: {end - start:.4f}秒")
    return wrapper

# 原有函数
def work():
    time.sleep(1) # 程序强制暂停1秒
    print("任务执行完毕")

# 手动装饰： 替换原函数
work = timer(work)
work()

任务执行完毕
运行耗时: 1.0005秒


### 三、@语法糖
`@装饰器名` 等价于 `函数 = 装饰器(函数)`

In [9]:
# 语法糖改写
@timer
def work():
    time.sleep(1) 
    print("任务执行完毕")

work()

任务执行完毕
运行耗时: 1.0012秒


### 四、兼容带参数、有返回值的原函数

In [12]:
# 通用写法
import time

def timer(func):
    # *args接收任意的位置参数，**kwargs接收任意关键字参数
    def wrapper(*args, **kwargs):
        start = time.time()
        # 接收原函数返回值
        res = func(*args, **kwargs)
        end = time.time()
        print(f"耗时：{end - start:.4f}秒")
        return res
    return wrapper

@timer
def add(a, b):
    time.sleep(0.5)
    return a + b

print(add(3, 5))

耗时：0.5008秒
8


#### 补充装饰器通用参数

In [ ]:
def wrapper(*args, **kwargs):
    # 能接收任意数量、任意类型参数
    pass

- `*args` --- 收集所有多余的 **位置参数**，打包成元组`tuple`
- `**kwargs` --- 收集所有多余的 **关键字参数**，打包成字典`dict`

### 五、问题：装饰后函数属性被篡改

In [14]:
print(add.__name__)
# 被装饰后的函数，函数名、文档注释都会变成 wrapper

wrapper


**解决：导入 functools.wraps**

In [ ]:
import time
from functools import wraps

def timer(func):
    @wraps(func)  # 复刻原函数信息
    def wrapper(*args, **kwargs):
        start = time.time()
        res = func(*args, **kwargs)
        print(f"耗时：{time.time()-start:.4f}")
        return res
    return wrapper

### 七、带参数的装饰器（双层嵌套）
- 需求：装饰器自身需要传入配置参数，格式再多套一层外层函数。**自定义打印日志前缀**

In [17]:
from functools import wraps

# 第一层：接收装饰器参数
def log(text):
    # 第二层：接收目标函数
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f"【{text}】开始执行")
            res = func(*args, **kwargs)
            print(f"【{text}】执行结束")
            return res
        return wrapper
    return decorator


# 调用：log("计算")先执行，返回decorator，再装饰add
@log("计算")
def add(a,b):
    return a+b

add(2,4)

【计算】开始执行
【计算】执行结束


6

### 八、常用实战场景
1. **计时统计**：查看函数运行速度
2. **日志记录**：函数调用记录、入参记录
3. **权限校验**：登录判断，未登录禁止执行函数
4. **缓存优化**：重复计算结果缓存（lru_cache 就是内置装饰器）
5. **异常捕获**：统一给函数加 try except

In [20]:
# 练习1
# 请设计一个decorator，它可作用于任何函数上，并打印该函数的执行时间
import time, functools

def metric(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        # TODO
        start = time.time()
        res = fn(*args, **kwargs)
        end = time.time()
        print('%s executed in %s ms' % (fn.__name__, (end - start)* 1000))
        return res
    return wrapper

# 测试
@metric
def fast(x, y):
    time.sleep(0.0012)
    return x + y;

@metric
def slow(x, y, z):
    time.sleep(0.1234)
    return x * y * z;

f = fast(11, 22)
s = slow(11, 22, 33)
if f != 33:
    print('测试失败!')
elif s != 7986:
    print('测试失败!')


fast executed in 2.651691436767578 ms
slow executed in 123.92854690551758 ms


In [21]:
# 练习2
# 同一个 @log 两种用法兼容： 无参数写法：@log， 带参数写法：@log('execute')
import functools

def log(text=None):
    # 如果传入的是字符串：带参数用法 @log("xxx")
    if isinstance(text, str):
        def decorator(func):
            @functools.wraps(func)
            def wrapper(*args, **kwargs):
                print(f"begin call {text}")
                res = func(*args, **kwargs)
                print(f"end call {text}")
                return res
            return wrapper
        return decorator   # 最后返回decorator 
    # 没有传字符串，text是函数：无参数用法 @log
    else:
        func = text
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            print("begin call")
            res = func(*args, **kwargs)
            print("end call")
            return res
        return wrapper   # 最后返回wrapper

# 调用
# 无参数调用：@log
@log
def f1():
    print("函数f1运行中")

# 执行函数
f1()


# 带参数调用：@log("自定义文本")
@log('execute')
def f2():
    print("函数f2运行中")

f2()

begin call
函数f1运行中
end call
begin call execute
函数f2运行中
end call execute


## 五、偏函数
- 即`functools.partial`,把一个函数的某些参数给固定住（也就是设置默认值），返回一个新的函数，调用这个新函数会更简单

In [23]:
# 比如想新定义一个int2函数，默认转换为二进制
import functools
int2 = functools.partial(int,base=2)
print(int2('1000000'))

# 但int2的base值仍可以设置成其他
print(int2('1000000', base=10))

64
1000000


- 最后，创建偏函数时，实际上可以接收函数对象、*args 和 **kw这3个参数

In [24]:
# 当传入：
max2 = functools.partial(max, 10)
# 实际上会把10作为*args的一部分自动加到左边，也就是：把10传入调用时传入得序列里一起比较
max2(5, 6, 7)

10